In [8]:
import time
from pymongo import MongoClient

NOMBRE_GRUPO = "Energia y sustentabilidad"
CATEGORIA_ACTUAL = "Travel"

client = MongoClient('mongodb://database:27017/')
db = client['proyecto_bigdata']
coleccion = db['datos_libros']

print("CONEXION EXITOSA")

lista_libros = [
    {"titulo": "Libro 1", "precio": "45.17"},
    {"titulo": "Libro 2", "precio": "49.43"}
]

for libro in lista_libros:
    registro = {
        "item": libro["titulo"],
        "valor": float(libro["precio"]),
        "categoria": CATEGORIA_ACTUAL,
        "grupo": NOMBRE_GRUPO,
        "fecha_captura": time.strftime("%Y-%m-%d %H:%M:%S")
    }
    coleccion.insert_one(registro)

print("DATOS INSERTADOS")

CONEXION EXITOSA
DATOS INSERTADOS


In [3]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("Analisis_Categorias") \
    .config("spark.mongodb.read.connection.uri", "mongodb://database:27017/proyecto_bigdata.datos_libros") \
    .config("spark.mongodb.write.connection.uri", "mongodb://database:27017/proyecto_bigdata.datos_libros") \
    .config("spark.jars.packages", "org.mongodb.spark:mongo-spark-connector_2.12:10.3.0") \
    .getOrCreate()

In [4]:
df = spark.read.format("mongodb").load()

df.show()

+--------------------+---------+-------------------+--------------------+-------+-----+
|                 _id|categoria|      fecha_captura|               grupo|   item|valor|
+--------------------+---------+-------------------+--------------------+-------+-----+
|69e7026928033bde8...|   Travel|2026-04-21 04:51:53|        G1_Ecommerce|Libro 1|45.17|
|69e7026928033bde8...|   Travel|2026-04-21 04:51:53|        G1_Ecommerce|Libro 2|49.43|
|69e7039228033bde8...|   Travel|2026-04-21 04:56:50|        G1_Ecommerce|Libro 1|45.17|
|69e7039228033bde8...|   Travel|2026-04-21 04:56:50|        G1_Ecommerce|Libro 2|49.43|
|69e7049baf1721877...|   Travel|2026-04-21 05:01:15|Energia y sustent...|Libro 1|45.17|
|69e7049baf1721877...|   Travel|2026-04-21 05:01:15|Energia y sustent...|Libro 2|49.43|
|69e70620af1721877...|   Travel|2026-04-21 05:07:44|Energia y sustent...|Libro 1|45.17|
|69e70620af1721877...|   Travel|2026-04-21 05:07:44|Energia y sustent...|Libro 2|49.43|
+--------------------+---------+

In [5]:
df.groupBy("categoria").count().show()

+---------+-----+
|categoria|count|
+---------+-----+
|   Travel|    8|
+---------+-----+



In [6]:
df.printSchema()
df.show(5)

root
 |-- _id: string (nullable = true)
 |-- categoria: string (nullable = true)
 |-- fecha_captura: string (nullable = true)
 |-- grupo: string (nullable = true)
 |-- item: string (nullable = true)
 |-- valor: double (nullable = true)

+--------------------+---------+-------------------+--------------------+-------+-----+
|                 _id|categoria|      fecha_captura|               grupo|   item|valor|
+--------------------+---------+-------------------+--------------------+-------+-----+
|69e7026928033bde8...|   Travel|2026-04-21 04:51:53|        G1_Ecommerce|Libro 1|45.17|
|69e7026928033bde8...|   Travel|2026-04-21 04:51:53|        G1_Ecommerce|Libro 2|49.43|
|69e7039228033bde8...|   Travel|2026-04-21 04:56:50|        G1_Ecommerce|Libro 1|45.17|
|69e7039228033bde8...|   Travel|2026-04-21 04:56:50|        G1_Ecommerce|Libro 2|49.43|
|69e7049baf1721877...|   Travel|2026-04-21 05:01:15|Energia y sustent...|Libro 1|45.17|
+--------------------+---------+-------------------+-------

In [7]:
from pyspark.sql.functions import col

df_filtrado = df.filter(
    (col("item").isNotNull()) & (col("valor") > 0)
)

print("Registros originales:", df.count())
print("Registros válidos:", df_filtrado.count())

Registros originales: 8
Registros válidos: 8


In [8]:
from pyspark.sql import functions as F

reporte_categorias = df.groupBy("categoria").agg(
    F.count("item").alias("cantidad_libros"),
    F.avg("valor").alias("precio_promedio"),
    F.min("valor").alias("precio_minimo"),
    F.max("valor").alias("precio_maximo")
).orderBy(F.desc("precio_promedio"))

reporte_categorias.show()

+---------+---------------+------------------+-------------+-------------+
|categoria|cantidad_libros|   precio_promedio|precio_minimo|precio_maximo|
+---------+---------------+------------------+-------------+-------------+
|   Travel|              8|47.300000000000004|        45.17|        49.43|
+---------+---------------+------------------+-------------+-------------+



In [9]:
from pyspark.sql import functions as F

ganador = df.orderBy(F.desc("valor")).limit(1)

ganador.select("item", "valor", "grupo", "fecha_captura").show()

+-------+-----+------------+-------------------+
|   item|valor|       grupo|      fecha_captura|
+-------+-----+------------+-------------------+
|Libro 2|49.43|G1_Ecommerce|2026-04-21 04:51:53|
+-------+-----+------------+-------------------+



In [11]:
ticket_salida = df.filter(F.col("grupo") == "Energia y sustentabilidad") \
    .groupBy("categoria") \
    .agg(
        F.count("item").alias("total_libros"),
        F.round(F.avg("valor"), 2).alias("precio_medio"),
        F.max("fecha_captura").alias("ultima_sincronizacion")
    )

print("--- TICKET DE SALIDA: Energia y sustentabilidad ---")
ticket_salida.show()

--- TICKET DE SALIDA: Energia y sustentabilidad ---
+---------+------------+------------+---------------------+
|categoria|total_libros|precio_medio|ultima_sincronizacion|
+---------+------------+------------+---------------------+
|   Travel|           4|        47.3|  2026-04-21 05:07:44|
+---------+------------+------------+---------------------+

